# mPES — Bayesian Optimization Monitor

**Dashboard de salud y progreso** para los runs de optimización corriendo en Colab Pro+.

- **Cell 2** — Snapshot único (todos los paquetes seleccionados)
- **Cell 3** — Panel auto-refrescante (un paquete, repinta cada N segundos)
- **Cell 4** — Detalles de un trial específico
- **Cell 5** — Optuna Dashboard (UI web interactiva)

Solo lectura: ejecutarlo **no interfiere** con la optimización en curso.


In [ ]:
# ────────────────────────────────────────────────────────────────────
# Cell 1 — PARAMETERS
# ────────────────────────────────────────────────────────────────────
PKGS_TO_CHECK = 'pes_rdqn'                    # qué paquetes monitorear
RUN_DATE      = '2026-04-29'                 # fecha del run (YYYY-MM-DD)
N_TARGET      = 20                          # nº trials objetivo (para ETA)
TAIL_LINES    = 15                           # líneas de log a mostrar
DRIVE_ROOT    = '/content/drive/MyDrive/mPES'
GIT_REPO      = 'https://github.com/Maximiliano0/mPES_2026.git'  # ← cambiar
GIT_BRANCH    = 'drl_models'
REPO_DIR      = '/content/Win_mPES'


## Setup (una vez por sesión)


In [ ]:
# ────────────────────────────────────────────────────────────────────
# Cell 1.5 — SETUP
# ────────────────────────────────────────────────────────────────────
import os, subprocess
from google.colab import drive

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--branch', GIT_BRANCH, GIT_REPO, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)

try:
    import optuna  # noqa: F401
except ImportError:
    subprocess.run(['pip', 'install', '-q', 'optuna'], check=True)

print(f'Repo  : {REPO_DIR}')
print(f'Drive : {DRIVE_ROOT}')
print('Setup OK.')


Mounted at /content/drive
Repo  : /content/Win_mPES
Drive : /content/drive/MyDrive/mPES
Setup OK.


## Cell 2 — Snapshot de todos los paquetes


In [10]:
# ────────────────────────────────────────────────────────────────────
# Cell 2 — SNAPSHOT ÚNICO (inline, sin subprocess)
# ────────────────────────────────────────────────────────────────────
import os, datetime, optuna

def _tail(p, n):
    if not os.path.isfile(p): return [f'(missing) {p}']
    try: return open(p, errors='replace').readlines()[-n:]
    except Exception as e: return [f'(error: {e})']

def report(pkg, date):
    pkg_dir = f'{DRIVE_ROOT}/{pkg}/{date}_BAYESIAN_OPT'
    db = f'{pkg_dir}/optuna_study_{date}.db'
    print('=' * 72); print(f'  {pkg}  |  run {date}'); print('=' * 72)
    print(f'  dir : {pkg_dir}')
    if not os.path.isfile(db):
        print(f'  DB  : NOT FOUND'); return
    storage = f'sqlite:///{db}'
    name = optuna.get_all_study_summaries(storage)[0].study_name
    s = optuna.load_study(study_name=name, storage=storage)
    by = {}
    for t in s.trials: by.setdefault(t.state.name, []).append(t)
    done = by.get('COMPLETE', []); run = by.get('RUNNING', []); fail = by.get('FAIL', [])
    print(f'  study     : {name}')
    print(f'  trials    : total={len(s.trials)}  COMPLETE={len(done)}  RUNNING={len(run)}  FAIL={len(fail)}')
    if done: print(f'  best value: {s.best_value:.6f}  (trial #{s.best_trial.number})')
    if len(done) >= 2:
        d = sorted(done, key=lambda t: t.datetime_complete or datetime.datetime.min)
        if d[0].datetime_start and d[-1].datetime_complete:
            span = (d[-1].datetime_complete - d[0].datetime_start).total_seconds()
            if span > 0:
                rate_h = len(done) / span * 3600
                rem = max(N_TARGET - len(done), 0)
                eta = datetime.timedelta(seconds=int(rem * span / len(done))) if rate_h else 'n/a'
                print(f'  throughput: {rate_h:.2f} trials/h  (avg {span/len(done):.1f}s/trial)')
                print(f'  ETA to {N_TARGET}: {eta}  ({rem} trials remaining)')
    print('-' * 72); print(f'  Last {TAIL_LINES} lines of bayesian_opt.log:')
    for ln in _tail(f'{pkg_dir}/bayesian_opt.log', TAIL_LINES): print('   ', ln.rstrip())
    err = [ln for ln in _tail(f'{pkg_dir}/bayesian_opt_err.log', TAIL_LINES) if ln.strip()]
    if err:
        print('-' * 72); print(f'  bayesian_opt_err.log:')
        for ln in err: print('   ', ln.rstrip())
    print()

report(PKGS_TO_CHECK, RUN_DATE)


  pes_rdqn  |  run 2026-04-29
  dir : /content/drive/MyDrive/mPES/pes_rdqn/2026-04-29_BAYESIAN_OPT
  study     : rdqn_opt_2026-04-29
  trials    : total=20  COMPLETE=20  RUNNING=0  FAIL=0
  best value: 0.925985  (trial #14)
  throughput: 1.29 trials/h  (avg 2800.8s/trial)
  ETA to 20: 0:00:00  (0 trials remaining)
------------------------------------------------------------------------
  Last 15 lines of bayesian_opt.log:
    
    --------------------------------------------------------------------------------
      Best RDQN Model
    --------------------------------------------------------------------------------
    
    ℹ Retraining best hyperparameters at full RDQN_EPISODES=175,000 (opt-trial used 30000 episodes)...
    Episode 10000 Average Reward: -140.3985895635458
    Episode 20000 Average Reward: -137.03426004112396
    Episode 30000 Average Reward: -133.45827881609952
    Episode 40000 Average Reward: -120.16028828006992
    Episode 50000 Average Reward: -110.26908714924869


## Cell 3 — Panel auto-refrescante (instantáneo, sin subprocess)
Repinta cada `REFRESH_SECS` segundos. Para detener: pulsa **■ Stop** de la celda.


In [9]:
# ────────────────────────────────────────────────────────────────────
# Cell 3 — PANEL AUTO-REFRESCANTE (in-cell, sin subprocess)
# Necesita haber corrido Cell 2 primero (define la función report).
# ────────────────────────────────────────────────────────────────────
WATCH_PKG     = 'pes_rdqn'        # paquete a observar
REFRESH_SECS  = 30                # segundos entre repintados

import time, datetime
from IPython.display import clear_output

try:
    while True:
        clear_output(wait=True)
        ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        print(f'[refresh every {REFRESH_SECS}s — last update {ts}]\n')
        report(WATCH_PKG, RUN_DATE)
        time.sleep(REFRESH_SECS)
except KeyboardInterrupt:
    print('\n[stopped]')


[refresh every 30s — last update 2026-04-30 16:52:48]

  pes_rdqn  |  run 2026-04-29
  dir : /content/drive/MyDrive/mPES/pes_rdqn/2026-04-29_BAYESIAN_OPT
  study     : rdqn_opt_2026-04-29
  trials    : total=20  COMPLETE=20  RUNNING=0  FAIL=0
  best value: 0.925985  (trial #14)
  throughput: 1.29 trials/h  (avg 2800.8s/trial)
  ETA to 20: 0:00:00  (0 trials remaining)
------------------------------------------------------------------------
  Last 15 lines of bayesian_opt.log:
    
    --------------------------------------------------------------------------------
      Best RDQN Model
    --------------------------------------------------------------------------------
    
    ℹ Retraining best hyperparameters at full RDQN_EPISODES=175,000 (opt-trial used 30000 episodes)...
    Episode 10000 Average Reward: -140.3985895635458
    Episode 20000 Average Reward: -137.03426004112396
    Episode 30000 Average Reward: -133.45827881609952
    Episode 40000 Average Reward: -120.16028828006992

## Inspect RDQN Training Time
Let's check the Optuna database for the trial hyperparameters and see if we can identify why it's taking so long.

In [11]:
import optuna
import os
import pandas as pd

pkg = 'pes_rdqn'
date = RUN_DATE
pkg_dir = f'{DRIVE_ROOT}/{pkg}/{date}_BAYESIAN_OPT'
db_path = f'{pkg_dir}/optuna_study_{date}.db'

if os.path.exists(db_path):
    storage = f'sqlite:///{db_path}'
    study_name = optuna.get_all_study_summaries(storage)[0].study_name
    study = optuna.load_study(study_name=study_name, storage=storage)

    print(f"Study: {study_name}")

    # Get all trials into a dataframe
    df_trials = study.trials_dataframe()

    if not df_trials.empty:
        display(df_trials)

        # Let's see the params for running trials
        running_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.RUNNING]
        if running_trials:
            print("\nCurrently Running Trial Parameters:")
            for t in running_trials:
                print(f"Trial #{t.number}:")
                for k, v in t.params.items():
                    print(f"  {k}: {v}")
    else:
        print("No trials found in the database.")
else:
    print(f"Database not found at {db_path}")

Study: rdqn_opt_2026-04-29


,number,value,datetime_start,datetime_complete,duration,params_batch_size,params_buffer_size,params_discount_factor,params_epsilon_initial,params_epsilon_min,...,params_use_pbrs,params_warmup_ratio,user_attrs_hidden_units,user_attrs_max_perf,user_attrs_mean_perf,user_attrs_min_perf,user_attrs_std_perf,user_attrs_trial_seed,system_attrs_fixed_params,state
0,0,0.900763,2026-04-29 22:57:30.072733,2026-04-30 00:20:05.891239,0 days 01:22:35.818506,128,20000,0.963424,0.962734,0.069147,...,True,0.277903,[64],0.977862,0.900763,0.749019,0.052834,43,"{'learning_rate': 0.0015083436603048935, 'disc...",COMPLETE
1,1,0.911139,2026-04-30 00:20:05.923851,2026-04-30 01:30:05.742053,0 days 01:09:59.818202,128,40000,0.924356,0.973235,0.124212,...,False,0.164017,[64],1.000000,0.911139,0.776788,0.049564,44,NaN,COMPLETE
2,2,0.859681,2026-04-30 01:30:05.767813,2026-04-30 02:15:19.127075,0 days 00:45:13.359262,64,40000,0.991166,0.993126,0.163595,...,False,0.243783,"[64, 64]",1.000000,0.859681,0.634749,0.097496,45,NaN,COMPLETE
3,3,0.898087,2026-04-30 02:15:19.158340,2026-04-30 02:56:38.087496,0 days 00:41:18.929156,32,90000,0.949151,0.854270,0.167460,...,True,0.078967,[128],1.000000,0.898087,0.771884,0.061042,46,NaN,COMPLETE
4,4,0.918227,2026-04-30 02:56:38.118433,2026-04-30 04:14:12.556256,0 days 01:17:34.437823,32,70000,0.986541,0.894443,0.032723,...,True,0.238888,"[96, 96]",0.985469,0.918227,0.773816,0.038239,47,NaN,COMPLETE
5,5,0.906373,2026-04-30 04:14:12.583207,2026-04-30 04:41:44.143638,0 days 00:27:31.560431,128,60000,0.980275,0.837314,0.179586,...,False,0.285727,[64],0.996303,0.906373,0.817533,0.048516,48,NaN,COMPLETE
6,6,0.893153,2026-04-30 04:41:44.175217,2026-04-30 05:56:02.995110,0 days 01:14:18.819893,128,80000,0.942566,0.856968,0.017009,...,True,0.208382,[128],0.996303,0.893153,0.763798,0.053106,49,NaN,COMPLETE
7,7,0.885381,2026-04-30 05:56:03.025357,2026-04-30 06:52:55.531163,0 days 00:56:52.505806,128,40000,0.921244,0.902419,0.053034,...,True,0.073276,[128],1.000000,0.885381,0.717130,0.066428,50,NaN,COMPLETE
8,8,0.916859,2026-04-30 06:52:55.559613,2026-04-30 07:23:28.250227,0 days 00:30:32.690614,256,70000,0.978491,0.928406,0.025987,...,False,0.236623,"[32, 32]",1.000000,0.916859,0.722725,0.061901,51,NaN,COMPLETE
9,9,0.866159,2026-04-30 07:23:28.277222,2026-04-30 07:36:50.379537,0 days 00:13:22.102315,32,30000,0.992976,0.878620,0.179489,...,True,0.282080,[32],1.000000,0.866159,0.610843,0.088524,52,NaN,COMPLETE
